In [2]:

import pandas as pd
from pathlib import Path

#Import "pandas" could not be resolved from source


#Load the customer reviews dataset

file_path = Path("Reviews.csv")

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

print("\nColumn names:")

print(df.columns.tolist())

Dataset shape: (6000, 16)

Column names:
['Customer_ID', 'review_text', 'sentiment_label', 'satisfaction_score', 'preferred_category', 'preferred_channel', 'customer_value_segment', 'customer_tenure_months', 'orders_last_12m', 'days_since_last_purchase', 'support_tickets_12m', 'return_rate', 'churn', 'purchase_next_30_days', 'data_origin', 'split']


In [11]:
df.head()

,Customer_ID,review_text,sentiment_label,satisfaction_score,preferred_category,preferred_channel,customer_value_segment,customer_tenure_months,orders_last_12m,days_since_last_purchase,support_tickets_12m,return_rate,churn,purchase_next_30_days,data_origin,split
0,CUST009609,I am unhappy with my Sports & Fitness order an...,negative,2.00,Sports & Fitness,Website,Low Value,26,8,84,4,0.190,1,0,synthetic_generated_for_training,train
1,CUST004601,I was disappointed with my Electronics purchas...,negative,1.56,Electronics,Mobile App,Medium Value,10,7,63,4,0.000,1,1,synthetic_generated_for_training,test
2,CUST000722,I was disappointed with my Sports & Fitness pu...,negative,1.89,Sports & Fitness,Website,Low Value,12,6,73,6,0.119,1,0,synthetic_generated_for_training,train
3,CUST007179,The Books shopping experience needs improvemen...,negative,2.41,Books,Physical Store,Low Value,12,4,322,0,0.210,1,0,synthetic_generated_for_training,train
4,CUST002160,"My Sports & Fitness order was frustrating, and...",negative,2.46,Sports & Fitness,Marketplace,Medium Value,20,6,36,2,0.327,0,1,synthetic_generated_for_training,validation


In [3]:
print("\nFirst Five Reviews:")
display(df[["review_text", "sentiment_label", "split"]].head())


First Five Reviews:


,review_text,sentiment_label,split
0,I am unhappy with my Sports & Fitness order an...,negative,train
1,I was disappointed with my Electronics purchas...,negative,test
2,I was disappointed with my Sports & Fitness pu...,negative,train
3,The Books shopping experience needs improvemen...,negative,train
4,"My Sports & Fitness order was frustrating, and...",negative,validation


In [4]:
print("\nSentiment distribution:")
print(df["sentiment_label"].value_counts())


Sentiment distribution:
sentiment_label
positive    3287
neutral     1988
negative     725
Name: count, dtype: int64


In [5]:
print("\nMissing review text:", df["review_text"].isna().sum())
print("Missing sentiment label:", df["sentiment_label"].isna().sum())


Missing review text: 0
Missing sentiment label: 0


In [6]:
import torch

print(torch.__version__)
print("PyTorch is available!")

2.14.1+cpu
PyTorch is available!


In [7]:
import torch
from transformers import pipeline

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_analyzer = pipeline(
    task="sentiment-analysis",
    model=MODEL_NAME,
    framework="pt"
)

print("Sentiment model loaded successfully!")

c:\Users\hp\anaconda3\envs\regression_app\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hp\anaconda3\envs\regression_app\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this a

Sentiment model loaded successfully!


In [8]:
test_reviews = [
    "The product quality is excellent and delivery was fast",
    "The product arrived damaged and customer support was unhelpful",
    "The package arrived yesterday"
]

for review in test_reviews:
    result = sentiment_analyzer(review)[0]
    print("Review:", review)
    print("Sentiment:", result["label"])
    print("Score:", round (result["score"], 4))
    print("-"* 60)

Review: The product quality is excellent and delivery was fast
Sentiment: positive
Score: 0.9786
------------------------------------------------------------
Review: The product arrived damaged and customer support was unhelpful
Sentiment: negative
Score: 0.9376
------------------------------------------------------------
Review: The package arrived yesterday
Sentiment: positive
Score: 0.4977
------------------------------------------------------------


In [9]:
print("Dataset split distribution:")
print(df["split"].value_counts())

print("\nSentiment labels:")
print(df["sentiment_label"].value_counts())

Dataset split distribution:
split
train         4800
test           600
validation     600
Name: count, dtype: int64

Sentiment labels:
sentiment_label
positive    3287
neutral     1988
negative     725
Name: count, dtype: int64


In [10]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Select held-out test data only
test_df = df.loc[
    df["split"].str.lower() == "test",
    ["review_text", "sentiment_label"]
].dropna().copy()

print("Test records:", len(test_df))

# Predict in batches
predictions = sentiment_analyzer(
    test_df["review_text"].astype(str).tolist(),
    batch_size=16,
    truncation=True
)

Test records: 600


In [ ]:
test_df["predicted_sentiment"]
